<a href="https://colab.research.google.com/github/lennartvoelz/fine_tune_hf/blob/main/test_precision_functiongemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch
!pip install transformers
!pip install peft

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from peft import PeftModel
from google.colab import userdata
from datasets import load_dataset
from google.colab import drive

hf_token = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/data/"

max_seq_length = 768
BASE_MODEL = "google/functiongemma-270m-it"
ADAPTER_PATH = SAVE_DIR + "lora_dir/"

TEST_FP32 = True
TARGET_DTYPE = torch.float32 if TEST_FP32 else torch.bfloat16

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=TARGET_DTYPE,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    dtype=TARGET_DTYPE,
    autocast_adapter_dtype=False,
)

In [ ]:
def _escape(s):
    return f"<escape>{s}<escape>"

def format_argument(value, escape_keys=True):
    if isinstance(value, str):
        return _escape(value)
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, dict):
        parts = []
        for k in sorted(value.keys()):
            key_str = _escape(k) if escape_keys else k
            parts.append(f"{key_str}:{format_argument(value[k], escape_keys)}")
        return "{" + ",".join(parts) + "}"
    if isinstance(value, (list, tuple)):
        return "[" + ",".join(format_argument(v, escape_keys) for v in value) + "]"
    if value is None:
        return "null"
    return str(value)

def _format_required_list(required):
    return "[" + ",".join(_escape(r) for r in required) + "]"

def format_parameters(properties, _required=None):
    STANDARD_KEYS = {"description", "type", "properties", "required", "nullable"}
    parts = []
    for key in sorted(properties.keys()):
        if key in STANDARD_KEYS: continue
        prop = properties[key]
        if not isinstance(prop, dict): continue
        inner = []
        if "description" in prop:
            inner.append(f"description:{_escape(prop['description'])}")
        type_val = prop.get("type", "")
        if type_val.upper() == "STRING" and "enum" in prop:
            inner.append(f"enum:{format_argument(prop['enum'], escape_keys=True)}")
        elif type_val.upper() == "OBJECT":
            if "properties" in prop:
                inner.append("properties:{" + format_parameters(prop["properties"], prop.get("required")) + "}")
            if "required" in prop:
                inner.append(f"required:{_format_required_list(prop['required'])}")
        elif type_val.upper() == "ARRAY":
            if "items" in prop and isinstance(prop["items"], dict):
                items = prop["items"]
                items_parts = []
                for item_key in sorted(items.keys()):
                    item_value = items[item_key]
                    if item_key == "properties":
                        items_parts.append("properties:{" + format_parameters(item_value, items.get("required")) + "}")
                    elif item_key == "required":
                        items_parts.append(f"required:{_format_required_list(item_value)}")
                    elif item_key == "type":
                        if isinstance(item_value, str): items_parts.append(f"type:{_escape(item_value.upper())}")
                        elif isinstance(item_value, list): items_parts.append("type:[" + ",".join(_escape(v.upper()) for v in item_value) + "]")
                    else:
                        items_parts.append(f"{item_key}:{format_argument(item_value, escape_keys=True)}")
                inner.append("items:{" + ",".join(items_parts) + "}")
        if type_val:
            inner.append(f"type:{_escape(type_val.upper())}")
        parts.append(f"{key}:{{{','.join(inner)}}}")
    return ",".join(parts)

def format_tools(tools):
    if not tools: return ""
    result = ""
    for tool in tools:
        t = tool["function"] if "function" in tool else tool
        result += "<start_function_declaration>"
        params = t.get("parameters", {})

        decl = f"declaration:{t.get('name', '')}{{description:{_escape(t.get('description', ''))}"
        if params:
            decl += ",parameters:{"
            if "properties" in params: decl += "properties:{" + format_parameters(params["properties"], params.get("required")) + "}"
            if "required" in params: decl += f",required:{_format_required_list(params['required'])}"
            if "type" in params: decl += f",type:{_escape(params['type'].upper())}"
            decl += "}"
        decl += "}"

        result += decl + "<end_function_declaration>"
    return result

def build_inference_prompt(user_msg, tools_list):
    """Build the strict FunctionGemma generation prompt"""
    system_content = "You are a model that can do function calling with the following functions"
    tools_formatted = format_tools(tools_list) if tools_list else ""

    prompt = "<bos>"
    if tools_formatted:
        prompt += f"<start_of_turn>developer\n{system_content}{tools_formatted}<end_of_turn>\n"
    else:
        prompt += f"<start_of_turn>developer\n{system_content}<end_of_turn>\n"

    prompt += f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
    prompt += "<start_of_turn>model\n"
    return prompt

In [ ]:
raw_data = []
with open(SAVE_DIR+"examples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

# Pick an example
example = raw_data[28]
user_msg = next((msg["content"] for msg in example["messages"] if msg["role"] == "user"), "")
tools_list = example.get("tools", [])

# Build prompt
text = build_inference_prompt(user_msg, tools_list)

# Tokenize
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Setup stop tokens
eos_token_id = tokenizer.eos_token_id
start_resp_token_id = tokenizer.convert_tokens_to_ids("<start_function_response>")
stop_tokens = [eos_token_id, start_resp_token_id]

# Setup Streamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\n--- INFERENCE START ---\n")

# Important: We must use a context manager for proper fp16/fp32 inference evaluation
with torch.no_grad():
    # If using fp16 on Gemma 3 models, standard torch.autocast can sometimes cause infinite activations
    # So we strictly run it in the dtype the model is loaded in.
    _ = model.generate(
        **inputs,
        max_new_tokens=768,
        streamer=streamer,
        do_sample=False,
        temperature=0.0,
        eos_token_id=stop_tokens,
    )